In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "raw" / "cicids2017_cleaned.csv"
SCHEMA_DIR = ROOT / "data" / "schema"

print("Dataset path:", RAW_CSV)
print("Dataset exists:", RAW_CSV.exists())


In [ ]:

df = pd.read_csv(RAW_CSV, low_memory=False)

df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

display(df.head())


In [ ]:

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nAttack Type column exists:", "Attack Type" in df.columns)


In [ ]:

class_counts = df["Attack Type"].value_counts()
class_percentages = df["Attack Type"].value_counts(normalize=True) * 100

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(2)
})

print("Total attack classes:", df["Attack Type"].nunique())
print("\nClass distribution:")
display(class_distribution)


In [ ]:

numeric_df = df.select_dtypes(include=[np.number])

missing_values = df.isna().sum()
infinite_values = np.isinf(numeric_df).sum()

print("Total missing values:", missing_values.sum())
print("Total infinite values:", infinite_values.sum())

print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

print("\nColumns with infinite values:")
print(infinite_values[infinite_values > 0])


In [ ]:

duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Total duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")


In [ ]:

unique_counts = df.nunique(dropna=False)
constant_columns = unique_counts[unique_counts <= 1]

print("Total constant columns:", len(constant_columns))
print("\nConstant columns:")
print(constant_columns)


In [ ]:

numeric_df = df.select_dtypes(include=[np.number])

feature_ranges = pd.DataFrame({
    "Minimum": numeric_df.min(),
    "Maximum": numeric_df.max(),
    "Mean": numeric_df.mean(),
    "Standard Deviation": numeric_df.std()
})

display(feature_ranges.round(4))


In [ ]:

negative_counts = (numeric_df < 0).sum()
negative_columns = negative_counts[negative_counts > 0].sort_values(ascending=False)

negative_summary = pd.DataFrame({
    "Negative Count": negative_columns,
    "Percentage": (negative_columns / len(df) * 100).round(4),
    "Minimum Value": numeric_df[negative_columns.index].min()
})

display(negative_summary)

In [ ]:

window_columns = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]

check_columns = negative_columns.index.difference(window_columns)

suspicious_rows = (df[check_columns] < 0).any(axis=1)

print("Rows with suspicious negative values:", suspicious_rows.sum())
print("Percentage:", round(suspicious_rows.mean() * 100, 4), "%")

print("\nAttack class distribution of suspicious rows:")
print(df.loc[suspicious_rows, "Attack Type"].value_counts())


In [ ]:

duplicate_rows = df.duplicated(keep="first")

print("Total duplicate rows:", duplicate_rows.sum())

print("\nDuplicate rows by attack class:")
print(df.loc[duplicate_rows, "Attack Type"].value_counts())


In [ ]:

SCHEMA_DIR.mkdir(parents=True, exist_ok=True)

report = "# CICIDS2017 Class Distribution\n\n"
report += f"Total records: {len(df):,}\n\n"
report += f"Total classes: {df['Attack Type'].nunique()}\n\n"
report += "| Attack Type | Count | Percentage |\n"
report += "|---|---:|---:|\n"

for attack_type, row in class_distribution.iterrows():
    report += f"| {attack_type} | {int(row['Count']):,} | {row['Percentage']:.2f}% |\n"

report += "\nThe dataset is highly imbalanced, with Normal Traffic representing 83.11% of records.\n"

report_path = SCHEMA_DIR / "class_distribution.md"
report_path.write_text(report, encoding="utf-8")

print("Report saved:", report_path)
print("File exists:", report_path.exists())


In [ ]:

feature_columns = [col for col in df.columns if col != "Attack Type"]

schema = {
    "version": 1,
    "dataset": "CICIDS2017 Cleaned and Preprocessed",
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "label_column": "Attack Type",
    "label_classes": sorted(df["Attack Type"].unique().tolist()),
    "excluded_columns": [],
    "feature_count": len(feature_columns),
    "features": [
        {
            "name": col,
            "dtype": str(df[col].dtype),
            "description": ""
        }
        for col in feature_columns
    ],
    "scaler_file": "models/scaler.pkl",
    "label_encoder_file": "models/label_encoder.pkl"
}

schema_path = SCHEMA_DIR / "feature_schema.json"
schema_path.parent.mkdir(parents=True, exist_ok=True)

with schema_path.open("w", encoding="utf-8") as file:
    json.dump(schema, file, indent=4)

print("Schema updated:", schema_path)
print("Feature count:", schema["feature_count"])
print("Label column:", schema["label_column"])
print("Label classes:", schema["label_classes"])


In [ ]:

with schema_path.open("r", encoding="utf-8") as file:
    saved_schema = json.load(file)

print("Version:", saved_schema["version"])
print("Label column:", saved_schema["label_column"])
print("Feature count:", saved_schema["feature_count"])
print("Label classes:", saved_schema["label_classes"])
print("Excluded columns:", saved_schema["excluded_columns"])
print("Scaler file:", saved_schema["scaler_file"])
print("Label encoder file:", saved_schema["label_encoder_file"])
print("First five features:", saved_schema["features"][:5])


In [ ]:

audit = f"""# CICIDS2017 Data Quality Audit

## Dataset Overview
- Total records: {len(df):,}
- Total columns: {len(df.columns)}
- Input features: {len(feature_columns)}
- Target column: Attack Type
- Traffic classes: {df["Attack Type"].nunique()}

## Quality Checks
- Missing values: {int(missing_values.sum()):,}
- Infinite values: {int(infinite_values.sum()):,}
- Duplicate records: {int(duplicate_count):,}
- Constant columns: {len(constant_columns)}
- Records with suspicious negative values: {int(suspicious_rows.sum()):,}

## Findings
- Normal Traffic represents 83.11% of the dataset.
- All 161 duplicate records belong to Normal Traffic.
- Some timing, rate, header-length, and segment-size features contain negative values.
- TCP window-size features contain -1 values that may represent undefined measurements.
- Numeric features have substantially different ranges and do not appear standardized.

## Preprocessing Considerations
- Review suspicious negative values before deciding whether to remove or correct records.
- Evaluate duplicate removal before model training.
- Split the dataset before fitting a scaler.
- Fit scaling parameters using training data only.
- Apply SMOTE only to the training partition, if selected.
- Preserve the original CSV.
"""

audit_path = SCHEMA_DIR / "data_quality_audit.md"
audit_path.write_text(audit, encoding="utf-8")

print("Audit saved:", audit_path)
print("File exists:", audit_path.exists())


In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "raw" / "cicids2017_cleaned.csv"
SCHEMA_DIR = ROOT / "data" / "schema"

LABEL = "Attack Type"

df = pd.read_csv(RAW_CSV)
df.columns = df.columns.str.strip()

print("Dataset loaded:", df.shape)
print("Label exists:", LABEL in df.columns)

In [ ]:
rows_before = len(df)
columns_before = len(df.columns)
before_counts = df[LABEL].value_counts()

print("Rows before cleaning:", rows_before)
print("Columns before cleaning:", columns_before)
print(before_counts)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns

infinite_count = np.isinf(df[num_cols].to_numpy()).sum()

df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

bad_rows = df.isna().any(axis=1)

print("Infinite values replaced:", infinite_count)
print("Rows with missing values:", bad_rows.sum())
print(df.loc[bad_rows, LABEL].value_counts())

In [ ]:
feature_cols = [c for c in df.columns if c != LABEL]

print("Exact duplicates:", df.duplicated().sum())
print(df.loc[df.duplicated(), LABEL].value_counts())
print("Feature-only duplicates:", df.duplicated(subset=feature_cols).sum())

df = df.drop_duplicates().copy()
gc.collect()

In [ ]:
feature_cols = [c for c in df.columns if c != LABEL]

const_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]

print("Constant columns:", const_cols)

df = df.drop(columns=const_cols)

num_now = df.select_dtypes(include=[np.number]).columns

print("Non-numeric features:", [c for c in df.columns if c != LABEL and c not in num_now])
print("ID-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["flow id", "ip", "timestamp", "date"])])

In [ ]:
df[LABEL] = df[LABEL].astype(str).str.strip()

NORMAL = "Normal Traffic"

print(sorted(df[LABEL].unique()))
assert NORMAL in set(df[LABEL])

after_counts = df[LABEL].value_counts()

compare = pd.DataFrame({
    "before": before_counts,
    "after": after_counts
})

compare["removed"] = compare["before"] - compare["after"]
compare["removed_pct"] = (
    compare["removed"] / compare["before"] * 100
).round(3)

print("Rows:", rows_before, "->", len(df))
print("Columns:", columns_before, "->", len(df.columns))

compare

In [ ]:
schema_path = SCHEMA_DIR / "feature_schema.json"
schema = json.loads(schema_path.read_text(encoding="utf-8"))

schema["excluded_columns"] = const_cols
schema["features"] = [
    {"name": c, "dtype": str(df[c].dtype)}
    for c in df.columns
    if c != LABEL
]
schema["version"] += 1

schema_path.write_text(
    json.dumps(schema, indent=2),
    encoding="utf-8"
)

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

df.to_parquet(INTERIM / "cleaned.parquet", index=False)

print("Saved cleaned data:", df.shape)

del df
gc.collect()

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

raw = pd.read_csv(root / "data/raw/cicids2017_cleaned.csv", low_memory=False)
cleaned = pd.read_parquet(root / "data/interim/cleaned.parquet")

raw.columns = raw.columns.str.strip()

before = raw["Attack Type"].value_counts()
after = cleaned["Attack Type"].value_counts()
comparison = pd.DataFrame({"Before": before, "After": after}).fillna(0).astype(int)
comparison["Removed"] = comparison["Before"] - comparison["After"]

print("Rows before:", len(raw))
print("Rows after:", len(cleaned))
print("Columns before:", len(raw.columns))
print("Columns after:", len(cleaned.columns))
print("Missing values after:", int(cleaned.isna().sum().sum()))
print("Infinite values after:", int(np.isinf(cleaned.select_dtypes(include=[np.number]).to_numpy()).sum()))
print("Duplicates after:", int(cleaned.duplicated().sum()))
print("\nClass comparison:")
print(comparison.to_string())
